# Tugas Analisis Algoritma: Kruskal, Prim, dan Borůvka (MST)
Muhammad Alif Qadri (D082261018) | Algoritma Komputasi, Magister Teknik Informatika | No 10, pembanding ketiga: Borůvka

Notebook Tugas 2: implementasi, uji kasus kecil, graf acak, eksperimen waktu, dan grafik dengan kurva teori.
**Menjalankan:** Colab (**Runtime > Run all**, izinkan Drive) atau lokal (`jalankan_notebook.bat`, lalu **Run All**). Hasil tersimpan di `hasil/`, `grafik/`, `data/`, dan `lingkungan_dan_log.txt`.
**Pustaka:** `numpy`, `pandas`, `matplotlib` (`networkx` hanya verifikasi). Pustaka MST tidak dipakai di dalam algoritma.
**Penjelasan rinci** tiap sel, hitungan tangan, keputusan desain, dan draf laporan penggunaan AI ada di `PENJELASAN_NOTEBOOK.md` (folder yang sama).

## Sel 1. Pengaturan
`SEED` tetap supaya hasil bisa diulang. Tiga skenario: graf jarang, graf padat, graf jarang dengan bobot kembar. Graf padat sengaja lebih kecil karena jumlah sisinya tumbuh seperti n². **`MODE_CEPAT = True` hanya untuk uji coba, jangan dipakai di laporan.**

In [ ]:
# ===== SEL 1: PENGATURAN =====
# SEED, ULANGAN, dan SKENARIO hanya dipakai di Bagian C (graf acak). Bagian A dan B memakai data tetap.
SEED = 2026                    # angka awal pengacak (tetap, supaya hasil bisa diulang)
ULANGAN = 5                    # pengulangan pengukuran per ukuran (minimal 5)
ULANG_GRAF_BARU = False        # False: 5 ulangan memakai graf yang sama (mengukur gangguan waktu)
                               # True : tiap ulangan memakai graf berbeda (seed + nomor ulangan)
BOBOT_MAKS_BIASA = 1_000_000   # bobot acak 1..angka ini, bobot kembar jarang terjadi
BOBOT_MAKS_KEMBAR = 5          # bobot acak 1..5, bobot kembar sangat banyak
MODE_CEPAT = False             # True: ukuran kecil untuk uji coba cepat

SKENARIO = {
    "jarang": {
        "keterangan": "graf jarang, m = 3n",
        "ukuran": [4000, 8000, 16000, 32000, 64000],
        "jumlah_sisi": lambda n: 3 * n,
        "bobot_maks": BOBOT_MAKS_BIASA,
    },
    "padat": {
        "keterangan": "graf padat, m = n(n-1)/4 (separuh dari jumlah sisi maksimum)",
        "ukuran": [200, 400, 600, 800, 1000],
        "jumlah_sisi": lambda n: n * (n - 1) // 4,
        "bobot_maks": BOBOT_MAKS_BIASA,
    },
    "jarang_kembar": {
        "keterangan": "graf jarang, m = 3n, bobot 1..5 (banyak bobot kembar)",
        "ukuran": [4000, 8000, 16000, 32000, 64000],
        "jumlah_sisi": lambda n: 3 * n,
        "bobot_maks": BOBOT_MAKS_KEMBAR,
    },
}

if MODE_CEPAT:  # ukuran kecil, hanya untuk mencoba apakah semua sel berjalan
    SKENARIO["jarang"]["ukuran"] = [100, 200, 400, 800, 1600]
    SKENARIO["padat"]["ukuran"] = [30, 50, 70, 90, 110]
    SKENARIO["jarang_kembar"]["ukuran"] = [100, 200, 400, 800, 1600]

for nama, s in SKENARIO.items():
    print(f"{nama:14s}: {s['keterangan']}; ukuran n = {s['ukuran']}")
print(f"Ulangan per ukuran: {ULANGAN} | seed: {SEED}")

## Sel 2. Folder hasil dan log
Colab: memasang Drive dan menyimpan ke `My Drive/TUGAS BESAR PAK ABDI/HASIL_EKSPERIMEN`. Lokal: di samping notebook, atau isi `FOLDER_DRIVE_LOKAL` untuk folder Google Drive for Desktop. Semua keluaran teks juga disalin ke `lingkungan_dan_log.txt`.

In [ ]:
# ===== SEL 2: FOLDER HASIL (LOKAL ATAU GOOGLE DRIVE) =====
import os
import sys

# Isi hanya bila menjalankan di komputer sendiri DAN ingin menyimpan ke folder Google Drive yang
# disinkronkan Google Drive for Desktop, contoh Windows: r"G:\My Drive\TUGAS BESAR PAK ABDI\HASIL_EKSPERIMEN"
# Biarkan kosong untuk menyimpan di samping notebook ini.
FOLDER_DRIVE_LOKAL = ""
# Folder khusus di Google Drive saat dijalankan di Colab (di dalam My Drive, dipisah dengan "/").
NAMA_FOLDER_DRIVE = "TUGAS BESAR PAK ABDI/HASIL_EKSPERIMEN"

try:
    from google.colab import drive  # modul ini hanya ada di Google Colab
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    drive.mount("/content/drive")  # Colab meminta izin akses Drive di sini
    FOLDER_DASAR = os.path.join("/content/drive/MyDrive", *NAMA_FOLDER_DRIVE.split("/"))
elif FOLDER_DRIVE_LOKAL:
    FOLDER_DASAR = FOLDER_DRIVE_LOKAL
else:
    FOLDER_DASAR = os.getcwd()  # folder tempat notebook ini dijalankan

FOLDER_HASIL = os.path.join(FOLDER_DASAR, "hasil")
FOLDER_GRAFIK = os.path.join(FOLDER_DASAR, "grafik")
FOLDER_DATA = os.path.join(FOLDER_DASAR, "data")
for folder in (FOLDER_HASIL, FOLDER_GRAFIK, FOLDER_DATA):
    os.makedirs(folder, exist_ok=True)
    print("Folder siap:", folder)

# Data studi kasus: dibuat dari isi bawaan bila belum ada. Berkas yang sudah ada tidak pernah ditimpa,
# jadi angka biaya yang kamu isi aman bila kamu mengeditnya langsung di folder ini.
BERKAS_KASUS = os.path.join(FOLDER_DATA, "studi_kasus_gedung_unhas.csv")
CSV_BAWAAN = """gedung_a,gedung_b,biaya_juta,keterangan
Sipil,Industri,15,ILUSTRASI - ganti dengan data biaya sebenarnya
Elektro,Geologi,0,ILUSTRASI - ganti dengan data biaya sebenarnya
Arsitektur,Sipil,-5,ILUSTRASI - ganti dengan data biaya sebenarnya
Arsitektur,Elektro,18,ILUSTRASI - ganti dengan data biaya sebenarnya
Sipil,Elektro,-2,ILUSTRASI - ganti dengan data biaya sebenarnya
Industri,Geologi,25,ILUSTRASI - ganti dengan data biaya sebenarnya
"""
if not os.path.exists(BERKAS_KASUS):
    with open(BERKAS_KASUS, "w", newline="", encoding="utf-8") as berkas:
        berkas.write(CSV_BAWAAN)
    print("Data studi kasus dibuat dari isi bawaan (masih ILUSTRASI):", BERKAS_KASUS)


class _Salin:
    """Menyalin semua keluaran print ke berkas log, tanpa mengubah tampilan di notebook."""

    def __init__(self, asli, berkas):
        self.asli, self.berkas = asli, berkas

    def write(self, teks):
        self.berkas.write(teks)
        return self.asli.write(teks)

    def flush(self):
        self.berkas.flush()
        self.asli.flush()

    def __getattr__(self, nama):  # sifat lain (misalnya encoding) diteruskan ke keluaran asli
        return getattr(self.asli, nama)


if hasattr(sys.stdout, "berkas"):  # sel ini dijalankan ulang: lepas pembungkus lama dulu
    sys.stdout.berkas.close()
    sys.stdout = sys.stdout.asli
BERKAS_LOG = os.path.join(FOLDER_DASAR, "lingkungan_dan_log.txt")
sys.stdout = _Salin(sys.stdout, open(BERKAS_LOG, "w", encoding="utf-8"))

print("=== Log notebook: Tugas_MST_Lokal ===")
print("Log ditulis ke:", BERKAS_LOG)
print(f"Parameter: SEED = {SEED} | ULANGAN = {ULANGAN} | ULANG_GRAF_BARU = {ULANG_GRAF_BARU} | MODE_CEPAT = {MODE_CEPAT}")
for nama, s in SKENARIO.items():
    print(f"  skenario {nama:14s}: {s['keterangan']}; ukuran n = {s['ukuran']}")


## Sel 3. Pustaka dan lingkungan uji
Spesifikasi mesin dicatat untuk Bab 4.1. Di Colab, mesinnya virtual dan dipakai bersama.

In [ ]:
# ===== SEL 3: IMPOR PUSTAKA DAN LINGKUNGAN UJI =====
import csv
import heapq
import math
import platform
import random
import statistics
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python      :", sys.version.split()[0])
print("Sistem      :", platform.system(), platform.release())


def nama_prosesor():
    """Nama prosesor dari /proc/cpuinfo (Linux, termasuk Colab); cadangan: platform.processor()."""
    try:
        with open("/proc/cpuinfo", encoding="utf-8") as berkas:
            for baris in berkas:
                if baris.lower().startswith("model name"):
                    return baris.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "(tidak diketahui)"


print("Lingkungan  :", "Google Colab (mesin virtual yang dipakai bersama)" if DI_COLAB else "Komputer lokal")
print("Prosesor    :", nama_prosesor(), "| jumlah CPU:", os.cpu_count())
print("Folder hasil:", FOLDER_DASAR)
if MODE_CEPAT:
    print("PERINGATAN  : MODE_CEPAT = True, hasil ini hanya uji coba dan jangan dipakai di laporan.")
print("pandas      :", pd.__version__, "| numpy:", np.__version__)

## Sel 4. Kruskal dan Himpunan Terpisah
Urutkan sisi menaik (`sorted()`, O(m log m)), pasang satu per satu kecuali yang membentuk siklus (dideteksi `HimpunanTerpisah`).
**Sumber:** *union by rank* dan kompresi lintasan menurut Sanders *et al.* (2019, Bagian 11.4); *path halving* adalah pilihan penulis. Kode bukan salinan dari repositori atau forum.

In [ ]:
# ===== SEL 4: KRUSKAL DAN HIMPUNAN TERPISAH =====
class HimpunanTerpisah:
    """Disjoint Set (Union-Find) dengan union by rank dan path halving."""

    def __init__(self, jumlah_simpul):
        self.induk = list(range(jumlah_simpul))  # awalnya induk tiap simpul adalah dirinya
        self.peringkat = [0] * jumlah_simpul     # perkiraan tinggi pohon

    def cari_akar(self, simpul):
        """Mencari akar (ketua) himpunan dari simpul."""
        while self.induk[simpul] != simpul:
            self.induk[simpul] = self.induk[self.induk[simpul]]  # path halving
            simpul = self.induk[simpul]
        return simpul

    def gabung(self, simpul_a, simpul_b):
        """Gabungkan himpunan kedua simpul. False jika sudah satu himpunan (siklus)."""
        akar_a = self.cari_akar(simpul_a)
        akar_b = self.cari_akar(simpul_b)
        if akar_a == akar_b:
            return False
        if self.peringkat[akar_a] < self.peringkat[akar_b]:  # pohon pendek digantung ke pohon tinggi
            akar_a, akar_b = akar_b, akar_a
        self.induk[akar_b] = akar_a
        if self.peringkat[akar_a] == self.peringkat[akar_b]:
            self.peringkat[akar_a] += 1
        return True


def kruskal(jumlah_simpul, daftar_sisi):
    """Mengembalikan (sisi_mst, total_bobot). ValueError jika graf tidak terhubung."""
    # Urut menaik menurut bobot; bila sama, simpul terkecil dulu, lalu simpul terbesar.
    sisi_terurut = sorted(daftar_sisi, key=lambda sisi: (sisi[2], min(sisi[0], sisi[1]), max(sisi[0], sisi[1])))

    himpunan = HimpunanTerpisah(jumlah_simpul)
    sisi_mst = []
    total_bobot = 0

    for simpul_a, simpul_b, bobot in sisi_terurut:  # dari sisi paling ringan
        if himpunan.gabung(simpul_a, simpul_b):  # terima jika belum terhubung (tanpa siklus)
            sisi_mst.append((simpul_a, simpul_b, bobot))
            total_bobot += bobot
            if len(sisi_mst) == jumlah_simpul - 1:  # MST sudah lengkap
                break

    if len(sisi_mst) != jumlah_simpul - 1:
        raise ValueError("Graf tidak terhubung")
    return sisi_mst, total_bobot

## Sel 5. Prim
Tumbuhkan pohon dari satu simpul, ambil sisi termurah yang keluar dari pohon (`heapq`, varian *lazy*, O(m log m) = O(m log n)). Memakai daftar ketetanggaan (Bab 1). Pemutus seri sama dengan Kruskal.

In [ ]:
# ===== SEL 5: PRIM =====
def buat_daftar_tetangga(jumlah_simpul, daftar_sisi):
    """Ubah daftar sisi (simpul_a, simpul_b, bobot) menjadi daftar ketetanggaan."""
    daftar_tetangga = [[] for _ in range(jumlah_simpul)]
    for simpul_a, simpul_b, bobot in daftar_sisi:
        daftar_tetangga[simpul_a].append((simpul_b, bobot))
        daftar_tetangga[simpul_b].append((simpul_a, bobot))  # graf tak berarah
    return daftar_tetangga


def prim(jumlah_simpul, daftar_tetangga, simpul_awal=0):
    """Mengembalikan (sisi_mst, total_bobot). ValueError jika graf tidak terhubung."""
    sudah_masuk = [False] * jumlah_simpul  # simpul yang sudah ada di pohon
    sisi_mst = []
    total_bobot = 0

    # Pohon dimulai dari satu simpul; masukkan semua sisi yang keluar darinya.
    sudah_masuk[simpul_awal] = True
    antrean = []
    for simpul_tetangga, bobot in daftar_tetangga[simpul_awal]:
        heapq.heappush(antrean, (bobot, min(simpul_awal, simpul_tetangga), max(simpul_awal, simpul_tetangga)))

    while antrean and len(sisi_mst) < jumlah_simpul - 1:
        bobot, kecil, besar = heapq.heappop(antrean)  # sisi termurah yang menyentuh pohon
        if sudah_masuk[kecil] and sudah_masuk[besar]:  # kedua ujung sudah di pohon: siklus
            continue
        asal, tujuan = (kecil, besar) if sudah_masuk[kecil] else (besar, kecil)  # tujuan = ujung baru
        sudah_masuk[tujuan] = True  # simpul baru bergabung ke pohon
        sisi_mst.append((asal, tujuan, bobot))
        total_bobot += bobot
        for simpul_tetangga, bobot_baru in daftar_tetangga[tujuan]:
            if not sudah_masuk[simpul_tetangga]:
                heapq.heappush(antrean, (bobot_baru, min(tujuan, simpul_tetangga), max(tujuan, simpul_tetangga)))

    if len(sisi_mst) != jumlah_simpul - 1:
        raise ValueError("Graf tidak terhubung")
    return sisi_mst, total_bobot

## Sel 6. Borůvka (pembanding ketiga)
Tiap putaran, setiap komponen memilih sisi termurah yang keluar darinya, lalu semua pilihan dipasang. Putaran paling banyak log₂ n. Memakai `HimpunanTerpisah` dari Sel 4 dan pemutus seri yang sama.

In [ ]:
# ===== SEL 6: BORŮVKA =====
def kunci_sisi(sisi):
    """Kunci pembanding sisi: bobot, lalu simpul terkecil, lalu simpul terbesar (sama seperti Kruskal)."""
    return (sisi[2], min(sisi[0], sisi[1]), max(sisi[0], sisi[1]))


def boruvka(jumlah_simpul, daftar_sisi):
    """Mengembalikan (sisi_mst, total_bobot). ValueError jika graf tidak terhubung."""
    himpunan = HimpunanTerpisah(jumlah_simpul)
    sisi_mst = []
    total_bobot = 0
    jumlah_komponen = jumlah_simpul  # awalnya tiap simpul adalah satu komponen

    while jumlah_komponen > 1:
        termurah = {}  # akar komponen -> sisi termurah yang keluar dari komponen itu
        for simpul_a, simpul_b, bobot in daftar_sisi:
            akar_a = himpunan.cari_akar(simpul_a)
            akar_b = himpunan.cari_akar(simpul_b)
            if akar_a == akar_b:  # sisi di dalam satu komponen, abaikan
                continue
            sisi = (simpul_a, simpul_b, bobot)
            for akar in (akar_a, akar_b):  # sisi ini calon pilihan kedua komponen
                if akar not in termurah or kunci_sisi(sisi) < kunci_sisi(termurah[akar]):
                    termurah[akar] = sisi

        if not termurah:  # tidak ada sisi keluar: graf terpisah
            raise ValueError("Graf tidak terhubung")

        for simpul_a, simpul_b, bobot in termurah.values():  # gabungkan semua pilihan
            if himpunan.gabung(simpul_a, simpul_b):  # False jika sisi itu sudah dipasang
                sisi_mst.append((simpul_a, simpul_b, bobot))
                total_bobot += bobot
                jumlah_komponen -= 1

    return sisi_mst, total_bobot

# BAGIAN A. UJI KASUS KECIL (data tetap, tanpa seed)
Empat graf kecil yang jawabannya bisa dihitung dengan tangan. **Setiap kasus ada di selnya sendiri** dan dijalankan pada ketiga algoritma (Kruskal, Prim, Borůvka).

| Sel | Isi | Jawaban benar |
|---|---|---|
| Sel 7 | Persiapan: fungsi bantu (tidak menguji apa pun) | tidak ada |
| Sel 7.1 | Kasus 1: empat simpul biasa, satu sisi ditolak | total 7 |
| Sel 7.2 | Kasus 2: lima simpul, ada bobot 0 dan negatif | total 5 |
| Sel 7.3 | Kasus 3: bobot kembar | total 6 |
| Sel 7.4 | Kasus 4: graf tak terhubung | `ValueError` |
| Sel 7.5 | Rekap keempat kasus | semua lolos |

Jalankan Sel 7 lebih dulu, lalu Sel 7.1 sampai 7.5 (atau **Run all**). Hitungan tangan ada di `PENJELASAN_NOTEBOOK.md` bagian 6.7. **Hitung ulang sendiri sebelum dikumpulkan.**

## Sel 7. Persiapan uji kasus kecil
Simpul diberi nomor 0, 1, 2, dan seterusnya. Sel ini hanya menyiapkan fungsi bantu yang dipakai Sel 7.1 sampai 7.5. Graf pada bagian A ini murni contoh (simpul dan sisi), belum terkait data nyata. Data nyata baru dipakai pada studi kasus (Sel 7B).

In [ ]:
# ===== SEL 7: PERSIAPAN UJI KASUS KECIL =====
def jalankan_tiga_algoritma(jumlah_simpul, daftar_sisi):
    """Menjalankan ketiga algoritma pada graf yang sama. Mengembalikan {nama: (sisi_mst, total)}."""
    return {
        "Kruskal": kruskal(jumlah_simpul, daftar_sisi),
        "Prim": prim(jumlah_simpul, buat_daftar_tetangga(jumlah_simpul, daftar_sisi)),
        "Borůvka": boruvka(jumlah_simpul, daftar_sisi),
    }


def rapikan(sisi_mst):
    """Ubah sisi menjadi (kecil, besar, bobot) supaya arah sisi tidak memengaruhi perbandingan."""
    return {(min(a, b), max(a, b), w) for a, b, w in sisi_mst}


hasil_uji_kecil = {}  # nama kasus -> jenis hasil; diisi oleh Sel 7.1 sampai 7.4


def uji_graf_terhubung(kasus):
    """Jalankan satu kasus graf terhubung pada ketiga algoritma. Periksa total, jumlah sisi, dan himpunan sisi."""
    hasil = jalankan_tiga_algoritma(kasus["jumlah_simpul"], kasus["daftar_sisi"])
    for algoritma, (sisi_mst, total) in hasil.items():
        assert total == kasus["total_benar"], f"{kasus['nama']}: total {algoritma} = {total}, seharusnya {kasus['total_benar']}"
        assert len(sisi_mst) == kasus["jumlah_simpul"] - 1, f"{kasus['nama']}: jumlah sisi {algoritma} salah"
        assert rapikan(sisi_mst) == rapikan(kasus["sisi_benar"]), f"{kasus['nama']}: himpunan sisi {algoritma} salah"
    hasil_uji_kecil[kasus["nama"]] = "terhubung"
    print(f"[LOLOS] {kasus['nama']} | total = {kasus['total_benar']} | Kruskal, Prim, Borůvka sama")


def uji_graf_tak_terhubung(kasus):
    """Ketiga algoritma harus menolak graf tak terhubung dengan ValueError."""
    for algoritma, fungsi in [("Kruskal", lambda n, s: kruskal(n, s)),
                              ("Prim", lambda n, s: prim(n, buat_daftar_tetangga(n, s))),
                              ("Borůvka", lambda n, s: boruvka(n, s))]:
        try:
            fungsi(kasus["jumlah_simpul"], kasus["daftar_sisi"])
            raise AssertionError(f"{algoritma} seharusnya melempar ValueError")
        except ValueError:
            pass
    hasil_uji_kecil[kasus["nama"]] = "galat"
    print(f"[LOLOS] {kasus['nama']} | ketiganya melempar ValueError")

## Sel 7.1. Kasus 1: empat simpul biasa (satu sisi ditolak)
Graf: 4 simpul, 5 sisi, semua bobot berbeda. Graf ini sama dengan Tabel 3.1 di laporan. Jawaban benar: **total 7**, dengan sisi (0, 1) bobot 1, (1, 2) bobot 2, dan (2, 3) bobot 4. Yang diuji: sisi (0, 2) bobot 3 harus ditolak karena membentuk siklus.

In [ ]:
# ===== SEL 7.1: KASUS 1, EMPAT SIMPUL BIASA (SATU SISI DITOLAK) =====
KASUS_1 = {"nama": "Kasus 1: 4 simpul biasa (satu sisi ditolak)", "jumlah_simpul": 4,
           "daftar_sisi": [(2, 3, 4), (0, 2, 3), (1, 3, 5),
                           (0, 1, 1), (1, 2, 2)],
           "total_benar": 7,
           "sisi_benar": {(0, 1, 1), (1, 2, 2), (2, 3, 4)}}
uji_graf_terhubung(KASUS_1)

## Sel 7.2. Kasus 2: lima simpul, ada bobot 0 dan negatif
Graf: 5 simpul, 7 sisi, semua bobot berbeda, memuat satu sisi berbobot 0 dan satu sisi berbobot negatif (-2). Jawaban benar: **total 5**, dengan sisi (1, 2) bobot -2, (0, 1) bobot 0, (2, 3) bobot 3, dan (0, 4) bobot 4. Yang diuji: bobot 0 dan bobot negatif diperlakukan sama seperti bobot lain (yang menentukan hanya urutannya), dan sisi (0, 2) bobot 1 harus ditolak karena simpul 0 dan simpul 2 sudah tersambung lewat simpul 1.

In [ ]:
# ===== SEL 7.2: KASUS 2, LIMA SIMPUL DENGAN BOBOT 0 DAN NEGATIF =====
KASUS_2 = {"nama": "Kasus 2: 5 simpul, bobot 0 dan negatif", "jumlah_simpul": 5,
           "daftar_sisi": [(0, 1, 0), (1, 2, -2), (0, 2, 1),
                           (2, 3, 3), (0, 4, 4), (1, 3, 5),
                           (4, 1, 6)],
           "total_benar": 5,
           "sisi_benar": {(0, 1, 0), (1, 2, -2), (2, 3, 3), (0, 4, 4)}}
uji_graf_terhubung(KASUS_2)

## Sel 7.3. Kasus 3: bobot kembar
Graf: 4 simpul, 5 sisi, empat di antaranya berbobot 2 dan membentuk lingkaran. Graf ini sama dengan Tabel 3.4 di laporan. Jawaban benar: **total 6**, dengan himpunan sisi (0, 1), (0, 3), dan (1, 2). Yang diuji: aturan pemutus seri (bobot, simpul terkecil, simpul terbesar), sehingga ketiga algoritma harus memilih **himpunan sisi yang sama** dari empat MST yang mungkin.

In [ ]:
# ===== SEL 7.3: KASUS 3, BOBOT KEMBAR (TOTAL DAN HIMPUNAN SISI) =====
KASUS_3 = {"nama": "Kasus 3: bobot kembar (total dan himpunan sisi)", "jumlah_simpul": 4,
           "daftar_sisi": [(0, 1, 2), (1, 2, 2), (2, 3, 2),
                           (3, 0, 2), (1, 3, 5)],
           "total_benar": 6,
           "sisi_benar": {(0, 1, 2), (0, 3, 2), (1, 2, 2)}}  # unik karena aturan pemutus seri
uji_graf_terhubung(KASUS_3)

## Sel 7.4. Kasus 4: graf tak terhubung
Graf: 4 simpul, hanya 2 sisi ((0, 1) dan (2, 3)), jadi terpisah menjadi dua bagian. Jawaban benar: **tidak ada MST**. Yang diuji: ketiga algoritma harus melempar `ValueError`.

In [ ]:
# ===== SEL 7.4: KASUS 4, GRAF TAK TERHUBUNG (HARUS VALUEERROR) =====
KASUS_4 = {"nama": "Kasus 4: graf tak terhubung (harus ValueError)", "jumlah_simpul": 4,
           "daftar_sisi": [(0, 1, 1), (2, 3, 2)]}
uji_graf_tak_terhubung(KASUS_4)

## Sel 7.5. Rekap uji kasus kecil
Memastikan keempat kasus sudah dijalankan dan lolos. Sel 16 memakai angka `JUMLAH_KASUS_UJI` dari sini.

In [ ]:
# ===== SEL 7.5: REKAP UJI KASUS KECIL =====
assert len(hasil_uji_kecil) == 4, f"Baru {len(hasil_uji_kecil)} kasus yang dijalankan. Jalankan Sel 7.1 sampai 7.4 dulu."
jumlah_lolos = sum(1 for jenis in hasil_uji_kecil.values() if jenis == "terhubung")
JUMLAH_KASUS_UJI = jumlah_lolos + 1  # kasus graf terhubung + satu kasus graf tak terhubung
print(f"Semua {JUMLAH_KASUS_UJI} kasus uji lolos pada ketiga algoritma ({jumlah_lolos} kasus graf terhubung dan 1 kasus galat).")

# BAGIAN B. STUDI KASUS (data tetap, tanpa seed)
Penerapan pada jaringan kabel antar lima gedung Fakultas Teknik Unhas. Datanya dibaca dari berkas CSV, bukan dibangkitkan secara acak, sehingga **tidak memakai seed** dan hasilnya sama setiap dijalankan. Bagian ini tidak dipakai untuk mengukur waktu.

## Sel 7B. Studi kasus: jaringan kabel 5 gedung FT Unhas (bobot = biaya bersih)
Data tetap dari `data/studi_kasus_gedung_unhas.csv`, tanpa seed, bukan untuk mengukur waktu. Bobot sisi adalah **biaya bersih** pemasangan jalur dalam juta rupiah, yaitu biaya pemasangan dikurangi dana yang diterima untuk jalur itu (subsidi, hibah, atau dana pendamping). Biaya 0 berarti impas, dan biaya negatif berarti penerimaan bersih. **Selama kolom `keterangan` masih ILUSTRASI, angkanya contoh. Ganti dengan data biaya sebenarnya dan catat sumbernya.**

In [ ]:
# ===== SEL 7B: STUDI KASUS NYATA (JARINGAN ANTAR GEDUNG) =====
# BERKAS_KASUS ditetapkan di Sel 2 (folder data di FOLDER_DASAR)
with open(BERKAS_KASUS, newline="", encoding="utf-8") as berkas:
    baris_kasus = list(csv.DictReader(berkas))

nama_gedung = sorted({b[k] for b in baris_kasus for k in ("gedung_a", "gedung_b")})
nomor_gedung = {nama: i for i, nama in enumerate(nama_gedung)}  # nomor tetap menurut abjad
sisi_kasus = [(nomor_gedung[b["gedung_a"]], nomor_gedung[b["gedung_b"]], int(b["biaya_juta"])) for b in baris_kasus]

hasil_kasus = jalankan_tiga_algoritma(len(nama_gedung), sisi_kasus)
total_kasus = {a: t for a, (_, t) in hasil_kasus.items()}
assert len(set(total_kasus.values())) == 1, f"total bobot berbeda: {total_kasus}"
himpunan_kasus = {a: rapikan(s) for a, (s, _) in hasil_kasus.items()}
assert len({frozenset(h) for h in himpunan_kasus.values()}) == 1, "himpunan sisi ketiga algoritma berbeda"

sisi_mst_kasus, total_mst_kasus = hasil_kasus["Kruskal"]
print(f"Studi kasus: {len(nama_gedung)} gedung, {len(sisi_kasus)} jalur kabel yang mungkin")
for a, b, w in sorted(rapikan(sisi_mst_kasus), key=lambda s: (s[2], s[0], s[1])):
    print(f"  {nama_gedung[a]} - {nama_gedung[b]}: {w} juta")
print(f"Total biaya bersih minimum: {total_mst_kasus} juta (Kruskal, Prim, dan Borůvka sama)")
if any(b["keterangan"].upper().startswith("ILUSTRASI") for b in baris_kasus):
    print("PERINGATAN: data masih berlabel ILUSTRASI. Ganti dengan data biaya sebenarnya sebelum disebut data nyata.")


# BAGIAN C. EKSPERIMEN WAKTU (graf acak, memakai seed)
Mulai dari sini grafnya dibangkitkan secara acak dengan `random.Random(seed)`. `SEED`, `ULANGAN`, dan daftar `SKENARIO` (jarang, padat, jarang_kembar) diatur di **Sel 1**. Seed yang sama menghasilkan graf yang sama, sehingga eksperimen dapat diulang.

## Sel 8. Pembuat graf acak
Pohon acak dulu (pasti terhubung), lalu tambah sisi acak tanpa sisi ganda dan loop. Memakai `random.Random(seed)`: seed sama, graf sama.

In [ ]:
# ===== SEL 8: PEMBUAT GRAF ACAK =====
def buat_graf(jumlah_simpul, jumlah_sisi, bobot_maks, seed):
    """Graf acak terhubung tanpa sisi ganda dan tanpa loop. Mengembalikan daftar sisi."""
    maks_sisi = jumlah_simpul * (jumlah_simpul - 1) // 2
    if not (jumlah_simpul - 1 <= jumlah_sisi <= maks_sisi):
        raise ValueError(f"jumlah_sisi harus antara {jumlah_simpul - 1} dan {maks_sisi}")
    pengacak = random.Random(seed)
    sudah_ada = set()
    daftar_sisi = []

    # Tahap 1: pohon acak, supaya graf pasti terhubung.
    for simpul in range(1, jumlah_simpul):
        simpul_lain = pengacak.randrange(simpul)
        sudah_ada.add((simpul_lain, simpul))
        daftar_sisi.append((simpul_lain, simpul, pengacak.randint(1, bobot_maks)))

    # Tahap 2: tambah sisi acak sampai jumlahnya cukup.
    while len(daftar_sisi) < jumlah_sisi:
        a = pengacak.randrange(jumlah_simpul)
        b = pengacak.randrange(jumlah_simpul)
        if a == b:
            continue
        if a > b:
            a, b = b, a
        if (a, b) in sudah_ada:
            continue
        sudah_ada.add((a, b))
        daftar_sisi.append((a, b, pengacak.randint(1, bobot_maks)))

    pengacak.shuffle(daftar_sisi)
    return daftar_sisi

## Sel 9. Periksa pembuat graf
Seed sama menghasilkan graf sama, graf terhubung tanpa sisi ganda dan loop, jumlah sisi sesuai skenario.

In [ ]:
# ===== SEL 9: PERIKSA PEMBUAT GRAF =====
def graf_terhubung(jumlah_simpul, daftar_sisi):
    himpunan = HimpunanTerpisah(jumlah_simpul)
    for a, b, _ in daftar_sisi:
        himpunan.gabung(a, b)
    return len({himpunan.cari_akar(i) for i in range(jumlah_simpul)}) == 1


# (1) seed sama -> graf sama; seed beda -> graf beda
g1 = buat_graf(50, 120, 100, seed=SEED)
g2 = buat_graf(50, 120, 100, seed=SEED)
g3 = buat_graf(50, 120, 100, seed=SEED + 1)
assert g1 == g2 and g1 != g3
print("Seed tetap: seed sama menghasilkan graf identik, seed beda menghasilkan graf lain.")

# (2) terhubung, tanpa sisi ganda dan loop; (3) jumlah sisi sesuai skenario
for nama, s in SKENARIO.items():
    n = s["ukuran"][0]
    m = s["jumlah_sisi"](n)
    graf = buat_graf(n, m, s["bobot_maks"], seed=SEED)
    pasangan = {(min(a, b), max(a, b)) for a, b, _ in graf}
    assert graf_terhubung(n, graf), "graf tidak terhubung"
    assert len(pasangan) == len(graf) == m, "ada sisi ganda atau jumlah sisi salah"
    assert all(a != b for a, b, _ in graf), "ada loop"
    kepadatan = m / (n * (n - 1) / 2)
    print(f"{nama:14s}: n = {n:5d}, m = {m:6d}, kepadatan = {kepadatan:.4f} | terhubung, tanpa sisi ganda atau loop")

## Sel 10. Verifikasi dengan `networkx` (opsional)
`networkx` hanya pembanding, tidak di dalam algoritma: 300 graf acak kecil. Sel ini juga mencocokkan file `.py` dengan kode notebook (dilewati di Colab).

In [ ]:
# ===== SEL 10: VERIFIKASI TAMBAHAN DENGAN NETWORKX (OPSIONAL) =====
try:
    import networkx as nx
except ImportError:
    nx = None
    print("networkx tidak terpasang, sel ini dilewati (opsional).")

if nx is not None:
    pengacak = random.Random(SEED)
    jumlah_graf = 300
    for _ in range(jumlah_graf):
        n = pengacak.randint(2, 40)
        m = pengacak.randint(n - 1, min(n * (n - 1) // 2, 4 * n))
        graf = buat_graf(n, m, pengacak.choice([3, 10, 1000]), seed=pengacak.randint(0, 10**9))
        G = nx.Graph()
        G.add_nodes_from(range(n))
        G.add_weighted_edges_from(graf)
        total_referensi = nx.minimum_spanning_tree(G).size(weight="weight")
        for algoritma, (sisi_mst, total) in jalankan_tiga_algoritma(n, graf).items():
            assert total == total_referensi and len(sisi_mst) == n - 1, f"{algoritma} berbeda dari networkx"
    print(f"Lolos: total bobot Kruskal, Prim, dan Borůvka sama dengan networkx pada {jumlah_graf} graf acak.")

# --- Cek: file .py di folder Kruskal, Prim, dan Borůvka memberi hasil yang sama dengan kode di notebook ini ---
import importlib.util

folder_induk = os.path.dirname(FOLDER_DASAR)  # folder CODING
berkas_py = {
    "Kruskal": os.path.join(folder_induk, "Kruskal", "kruskal.py"),
    "Prim": os.path.join(folder_induk, "Prim", "prim.py"),
    "Borůvka": os.path.join(folder_induk, "Borůvka", "boruvka.py"),
}
if all(os.path.exists(p) for p in berkas_py.values()):
    def muat_modul(nama, jalur):
        spesifikasi = importlib.util.spec_from_file_location(nama, jalur)
        modul = importlib.util.module_from_spec(spesifikasi)
        spesifikasi.loader.exec_module(modul)
        return modul

    sys.path.insert(0, os.path.dirname(berkas_py["Kruskal"]))  # supaya boruvka.py menemukan kruskal.py
    modul_kruskal = muat_modul("kruskal_berkas", berkas_py["Kruskal"])
    modul_prim = muat_modul("prim_berkas", berkas_py["Prim"])
    modul_boruvka = muat_modul("boruvka_berkas", berkas_py["Borůvka"])
    pengacak = random.Random(SEED)
    for _ in range(100):
        n = pengacak.randint(2, 60)
        graf = buat_graf(n, pengacak.randint(n - 1, min(n * (n - 1) // 2, 4 * n)), pengacak.choice([3, 1000]), seed=pengacak.randint(0, 10**9))
        assert kruskal(n, graf) == modul_kruskal.kruskal(n, graf), "kruskal.py berbeda dari notebook"
        assert prim(n, buat_daftar_tetangga(n, graf)) == modul_prim.prim(n, modul_prim.buat_daftar_tetangga(n, graf)), "prim.py berbeda dari notebook"
        assert boruvka(n, graf) == modul_boruvka.boruvka(n, graf), "boruvka.py berbeda dari notebook"
    print("Lolos: file .py di folder Kruskal, Prim, dan Borůvka memberi hasil yang sama dengan kode di notebook ini (100 graf acak).")
else:
    print("Folder Kruskal, Prim, atau Borůvka tidak ditemukan di samping folder dasar (normal di Colab), pencocokan file .py dilewati.")

## Sel 11. Pengukuran waktu
`time.perf_counter()` hanya pada algoritma, graf yang sama untuk ketiganya, pemanasan sekali. Total bobot ketiganya harus sama. Waktu Prim dicatat dua kali: inti saja, dan dengan konversi daftar ketetanggaan.

In [ ]:
# ===== SEL 11: FUNGSI PENGUKURAN WAKTU =====
import gc

ALGORITMA_UTAMA = ["Kruskal", "Prim", "Borůvka"]
ALGORITMA_CSV = ["Kruskal", "Prim", "Prim (+konversi)", "Borůvka"]


def ukur(fungsi, *argumen):
    """Menjalankan fungsi sekali. Mengembalikan (hasil, waktu_detik)."""
    gc.collect()
    mulai = time.perf_counter()
    hasil = fungsi(*argumen)
    selesai = time.perf_counter()
    return hasil, selesai - mulai


def ukur_satu_ulangan(jumlah_simpul, daftar_sisi):
    """Mengukur ketiga algoritma pada satu graf. Mengembalikan {nama algoritma: (waktu, total_bobot)}."""
    daftar_tetangga, waktu_konversi = ukur(buat_daftar_tetangga, jumlah_simpul, daftar_sisi)
    (_, total_kruskal), waktu_kruskal = ukur(kruskal, jumlah_simpul, daftar_sisi)
    (_, total_prim), waktu_prim = ukur(prim, jumlah_simpul, daftar_tetangga)
    (_, total_boruvka), waktu_boruvka = ukur(boruvka, jumlah_simpul, daftar_sisi)
    assert total_kruskal == total_prim == total_boruvka, \
        f"Total bobot berbeda: Kruskal {total_kruskal}, Prim {total_prim}, Borůvka {total_boruvka}"
    return {
        "Kruskal": (waktu_kruskal, total_kruskal),
        "Prim": (waktu_prim, total_prim),
        "Prim (+konversi)": (waktu_prim + waktu_konversi, total_prim),
        "Borůvka": (waktu_boruvka, total_boruvka),
    }


# Contoh singkat satu pengukuran pada graf kecil
contoh_graf = buat_graf(1000, 3000, BOBOT_MAKS_BIASA, seed=SEED)
for algoritma, (detik, total) in ukur_satu_ulangan(1000, contoh_graf).items():
    print(f"{algoritma:18s}: {detik * 1000:8.2f} ms | total bobot = {total}")

## Sel 12. Jalankan eksperimen
Tiap skenario dan ukuran diukur `ULANGAN` kali, data mentah ke `hasil/eksperimen_mentah.csv`. Sekitar 1 sampai 3 menit.

In [ ]:
# ===== SEL 12: JALANKAN EKSPERIMEN DAN SIMPAN KE CSV =====
BERKAS_MENTAH = os.path.join(FOLDER_HASIL, "eksperimen_mentah.csv")
KOLOM = ["skenario", "n", "m", "algoritma", "ulangan", "waktu_detik", "total_bobot"]

waktu_mulai_eksperimen = time.perf_counter()
with open(BERKAS_MENTAH, "w", newline="", encoding="utf-8") as berkas:
    penulis = csv.writer(berkas)
    penulis.writerow(KOLOM)
    for nama_skenario, s in SKENARIO.items():
        for n in s["ukuran"]:
            m = s["jumlah_sisi"](n)
            graf_tetap = buat_graf(n, m, s["bobot_maks"], seed=SEED)  # dipakai bila ULANG_GRAF_BARU = False
            # pemanasan: hasilnya tidak dicatat
            ukur_satu_ulangan(n, graf_tetap)
            for ulangan in range(1, ULANGAN + 1):
                graf = buat_graf(n, m, s["bobot_maks"], seed=SEED + ulangan) if ULANG_GRAF_BARU else graf_tetap
                hasil = ukur_satu_ulangan(n, graf)
                for algoritma in ALGORITMA_CSV:
                    detik, total = hasil[algoritma]
                    penulis.writerow([nama_skenario, n, m, algoritma, ulangan, f"{detik:.6f}", total])
            print(f"selesai: {nama_skenario:14s} n = {n:6d}, m = {m:7d}")

lama = time.perf_counter() - waktu_mulai_eksperimen
print(f"\nEksperimen selesai dalam {lama:.1f} detik. Data mentah: {BERKAS_MENTAH}")

## Sel 13. Rata-rata dan simpangan baku
Simpangan baku sampel (dibagi n − 1), disimpan ke `hasil/ringkasan.csv`. Dengan `ULANG_GRAF_BARU = False`, ulangan memakai graf yang sama, jadi simpangan baku mengukur gangguan waktu, bukan variasi antar-graf.

In [ ]:
# ===== SEL 13: RINGKASAN RATA-RATA DAN SIMPANGAN BAKU =====
data_mentah = pd.read_csv(BERKAS_MENTAH)
ringkasan = (data_mentah
             .groupby(["skenario", "n", "m", "algoritma"], sort=False)["waktu_detik"]
             .agg(rata_rata="mean", simpangan_baku="std", jumlah_ulangan="count")
             .reset_index())
BERKAS_RINGKASAN = os.path.join(FOLDER_HASIL, "ringkasan.csv")
ringkasan.to_csv(BERKAS_RINGKASAN, index=False)

# Tampilan dalam milidetik agar mudah dibaca
tampil = ringkasan.copy()
tampil["rata_rata (ms)"] = (tampil["rata_rata"] * 1000).round(3)
tampil["simpangan_baku (ms)"] = (tampil["simpangan_baku"] * 1000).round(3)
for nama_skenario in SKENARIO:
    print(f"\n=== Skenario: {nama_skenario} ({SKENARIO[nama_skenario]['keterangan']}) ===")
    bagian = tampil[tampil["skenario"] == nama_skenario]
    print(bagian.pivot(index=["n", "m"], columns="algoritma", values="rata_rata (ms)")[ALGORITMA_CSV].to_string())
print("\nRingkasan lengkap disimpan di:", BERKAS_RINGKASAN)

## Sel 14. Kurva teori dan grafik
Kruskal c·m·log₂ m; Prim dan Borůvka c·m·log₂ n. Konstanta c dicocokkan, jadi yang diuji adalah **bentuk** kurva. Ubah `KURVA_TEORI` bila turunan di Bab 3 berbeda.

In [ ]:
# ===== SEL 14: KURVA TEORI DAN GRAFIK WAKTU =====
KURVA_TEORI = {  # f(n, m): bentuk teori, tanpa konstanta
    "Kruskal": lambda n, m: m * math.log2(m),
    "Prim": lambda n, m: m * math.log2(n),
    "Borůvka": lambda n, m: m * math.log2(n),
}
WARNA = {"Kruskal": "tab:blue", "Prim": "tab:green", "Borůvka": "tab:red", "Prim (+konversi)": "tab:olive"}
BERKAS_GRAFIK = {}


def cocokkan_konstanta(nilai_teori, waktu_ukur):
    """Konstanta c terbaik (kuadrat terkecil melalui titik asal) untuk waktu = c * teori."""
    nilai_teori, waktu_ukur = np.array(nilai_teori, float), np.array(waktu_ukur, float)
    return float((nilai_teori * waktu_ukur).sum() / (nilai_teori ** 2).sum())


for nama_skenario in SKENARIO:
    data = ringkasan[ringkasan["skenario"] == nama_skenario]
    fig, (ax_linear, ax_log) = plt.subplots(1, 2, figsize=(14, 5))
    for algoritma in ALGORITMA_CSV:
        d = data[data["algoritma"] == algoritma].sort_values("m")
        garis = "o:" if algoritma == "Prim (+konversi)" else "o-"
        for ax in (ax_linear, ax_log):
            ax.errorbar(d["m"], d["rata_rata"], yerr=d["simpangan_baku"], fmt=garis,
                        color=WARNA[algoritma], capsize=3, label=f"{algoritma} (ukur)")
        if algoritma in KURVA_TEORI:
            teori = [KURVA_TEORI[algoritma](n, m) for n, m in zip(d["n"], d["m"])]
            c = cocokkan_konstanta(teori, d["rata_rata"])
            m_halus = np.linspace(d["m"].min(), d["m"].max(), 200)
            n_halus = np.interp(m_halus, d["m"], d["n"])
            kurva = [c * KURVA_TEORI[algoritma](n, m) for n, m in zip(n_halus, m_halus)]
            for ax in (ax_linear, ax_log):
                ax.plot(m_halus, kurva, "--", color=WARNA[algoritma], alpha=0.7, label=f"{algoritma} (teori)")
    ax_linear.set_title(f"Skenario {nama_skenario}: skala linear")
    ax_log.set_title(f"Skenario {nama_skenario}: skala log-log")
    ax_log.set_xscale("log"); ax_log.set_yscale("log")
    for ax in (ax_linear, ax_log):
        ax.set_xlabel("jumlah sisi (m)"); ax.set_ylabel("waktu (detik)"); ax.grid(alpha=0.3)
    ax_linear.legend(fontsize=8)
    fig.suptitle(f"Waktu eksekusi vs ukuran graf ({SKENARIO[nama_skenario]['keterangan']})")
    fig.tight_layout()
    berkas = os.path.join(FOLDER_GRAFIK, f"waktu_{nama_skenario}.png")
    fig.savefig(berkas, dpi=150, bbox_inches="tight")
    BERKAS_GRAFIK[nama_skenario] = berkas
    plt.show()
print("Grafik disimpan di:", FOLDER_GRAFIK)

## Sel 15. Rasio waktu terhadap teori
Waktu ÷ teori, dinormalkan ke ukuran terkecil (1,00). Angka yang menyimpang dijelaskan di pembahasan. Hasil kurang bagus tetap didokumentasikan.

In [ ]:
# ===== SEL 15: TABEL RASIO TERHADAP TEORI =====
baris_rasio = []
for nama_skenario in SKENARIO:
    for algoritma in ALGORITMA_UTAMA:
        d = (ringkasan[(ringkasan["skenario"] == nama_skenario) & (ringkasan["algoritma"] == algoritma)]
             .sort_values("m").reset_index(drop=True))
        teori = np.array([KURVA_TEORI[algoritma](n, m) for n, m in zip(d["n"], d["m"])])
        rasio = d["rata_rata"].to_numpy() / teori
        rasio = rasio / rasio[0]  # dinormalkan terhadap ukuran terkecil
        for n, m, r in zip(d["n"], d["m"], rasio):
            baris_rasio.append({"skenario": nama_skenario, "algoritma": algoritma, "n": n, "m": m, "rasio": round(float(r), 3)})
tabel_rasio = pd.DataFrame(baris_rasio)
tabel_rasio.to_csv(os.path.join(FOLDER_HASIL, "rasio_terhadap_teori.csv"), index=False)
for nama_skenario in SKENARIO:
    print(f"\n=== Rasio waktu/teori, skenario {nama_skenario} (ukuran terkecil = 1,00) ===")
    print(tabel_rasio[tabel_rasio["skenario"] == nama_skenario]
          .pivot(index=["n", "m"], columns="algoritma", values="rasio")[ALGORITMA_UTAMA].to_string())

## Sel 16. Pemeriksaan kelengkapan syarat
**[OK]** berarti syarat jumlah dan berkas terpenuhi.

In [ ]:
# ===== SEL 16: PEMERIKSAAN KELENGKAPAN SYARAT =====
def periksa(nama, kondisi):
    print(("[OK]    " if kondisi else "[KURANG] ") + nama)
    return kondisi

semua = []
semua.append(periksa("Kruskal, Prim, dan Borůvka ada dan berjalan", all(callable(f) for f in (kruskal, prim, boruvka))))
semua.append(periksa(f"Minimal 3 kasus uji kecil (graf terhubung) lolos pada ketiga algoritma (ada {JUMLAH_KASUS_UJI - 1})", JUMLAH_KASUS_UJI - 1 >= 3))
semua.append(periksa(f"Seed tetap dipakai (SEED = {SEED})", isinstance(SEED, int)))
for nama_skenario, s in SKENARIO.items():
    semua.append(periksa(f"Skenario {nama_skenario}: minimal 5 ukuran berbeda ({len(set(s['ukuran']))} ukuran)", len(set(s["ukuran"])) >= 5))
semua.append(periksa(f"Minimal 5 ulangan per percobaan (ULANGAN = {ULANGAN})", ULANGAN >= 5))
semua.append(periksa("Setiap kombinasi punya >= 5 data (rata-rata dan simpangan baku valid)", (ringkasan["jumlah_ulangan"] >= 5).all()))
semua.append(periksa("CSV data mentah tersimpan", os.path.exists(BERKAS_MENTAH)))
semua.append(periksa("CSV ringkasan tersimpan", os.path.exists(BERKAS_RINGKASAN)))
semua.append(periksa("Grafik dengan kurva teori tersimpan untuk semua skenario", all(os.path.exists(p) for p in BERKAS_GRAFIK.values())))
semua.append(periksa("Pembanding ketiga (Borůvka) ikut diukur", "Borůvka" in set(data_mentah["algoritma"])))
print("\nHasil: SEMUA SYARAT KODE TERPENUHI" if all(semua) else "\nMasih ada yang kurang, lihat tanda [KURANG].")
print("Folder hasil:", FOLDER_DASAR)

## Sel 17. Catatan metodologi dan laporan penggunaan AI
Keputusan desain, batasan, sumber, dan **draf laporan penggunaan AI** ada di `PENJELASAN_NOTEBOOK.md` bagian 6.17. Tulis ulang draf itu dengan kata-katamu sendiri sebelum dikumpulkan.

## Sel 18. Simpan semua hasil (jalankan paling akhir)
Menutup log dan, di Colab, menyinkronkan lalu melepas Drive. Untuk menjalankan ulang, mulai dari Sel 2.

In [ ]:
# ===== SEL 18: PASTIKAN SEMUA HASIL TERSIMPAN =====
print("Folder dasar:", FOLDER_DASAR)
for folder in (FOLDER_DATA, FOLDER_HASIL, FOLDER_GRAFIK):
    for nama_berkas in sorted(os.listdir(folder)):
        jalur = os.path.join(folder, nama_berkas)
        if os.path.isfile(jalur):
            print(f"  {os.path.basename(folder)}/{nama_berkas}  ({os.path.getsize(jalur):,} byte)")
print("  lingkungan_dan_log.txt  (log ini, ditutup setelah sel ini)")

if hasattr(sys.stdout, "berkas"):  # tutup log dan kembalikan keluaran normal
    log_terbuka = sys.stdout.berkas
    sys.stdout = sys.stdout.asli
    log_terbuka.close()

if DI_COLAB:
    drive.flush_and_unmount()  # paksa semua berkas tertulis ke Google Drive
    print("Google Drive sudah disinkronkan dan dilepas. Semua hasil ada di folder:", NAMA_FOLDER_DRIVE)
else:
    print("Selesai. Hasil ada di:", FOLDER_DASAR)
